# 03 — 后端

**来源:** [LangChain Docs — Deep Agents / Backends](https://docs.langchain.com/oss/python/deepagents/backends)

Deep Agents 将文件系统暴露给 Agent，通过 `ls`、`read_file`、`write_file`、`edit_file`、`glob`、`grep` 等工具操作文件，所有操作经过可插拔的后端（Backend）。

`read_file` 跨所有后端原生支持图片文件（`.png`、`.jpg`、`.jpeg`、`.gif`、`.webp`），返回多模态内容块。Sandbox 和 `LocalShellBackend` 还提供 `execute` 工具。

本笔记涵盖：
- 各内置 Backend 的用途与用法
- StateBackend / FilesystemBackend / StoreBackend / ContextHubBackend / CompositeBackend
- LocalShellBackend 与 Sandbox
- 路由不同路径到不同后端
- 文件系统权限控制（Permissions）
- 自定义虚拟文件系统（S3 / Postgres）

## 1. 后端架构概览

```
Filesystem Tools → Backend → State（线程内）
                            → Filesystem（本地磁盘）
                            → Store（LangGraph 持久存储）
                            → ContextHub（LangSmith Hub）
                            → Sandbox（隔离环境）
                            → LocalShell（本地 Shell）
                            → Composite（路由组合）
                            → Custom（自定义）
```

| 后端 | 描述 |
|------|------|
| **StateBackend**（默认） | 线程级存储，持久化在 LangGraph state 中，跨轮次持久但线程间不共享 |
| **FilesystemBackend** | 真实文件系统访问，可设置 `root_dir` |
| **StoreBackend** | 基于 LangGraph BaseStore 的跨线程持久存储 |
| **ContextHubBackend** | 基于 LangSmith Hub repo 的持久化 |
| **Sandbox** | 隔离沙箱环境（Modal / Daytona / Deno / 本地 VFS） |
| **LocalShellBackend** | 直接在本机执行 Shell 命令 |
| **CompositeBackend** | 路由不同路径到不同后端，灵活性最高 |

## 2. 快速选择指南

| 场景 | 推荐后端 |
|------|----------|
| 默认（线程内 scratch pad） | `StateBackend()` |
| 本地文件系统访问 | `FilesystemBackend(root_dir=...)` |
| 跨线程持久化 | `StoreBackend()` + `store=` |
| LangSmith Hub 存储 | `ContextHubBackend("my-agent")` |
| 隔离代码执行 | `sandbox` |
| 本地开发 CLI | `LocalShellBackend(root_dir=".")` |
| 多源混合 | `CompositeBackend(default=..., routes={...})` |

## 3. StateBackend（默认）

框架默认使用 `StateBackend`，文件存储在 LangGraph Agent State 中，通过 checkpoint 跨轮次持久化，但线程间不共享。

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import StateBackend

# 默认方式（隐式使用 StateBackend）
agent = create_deep_agent(model="deepseek:deepseek-v4-flash")

# 显式指定（两者等价）
agent2 = create_deep_agent(
    model="deepseek:deepseek-v4-flash",
    backend=StateBackend(),
)

print("Agent 使用 StateBackend（线程内存储）。")
print("特点：跨轮次持久化、线程间不共享、子 Agent 间共享同一后端。")

Agent 使用 StateBackend（线程内存储）。
特点：跨轮次持久化、线程间不共享、子 Agent 间共享同一后端。


## 4. FilesystemBackend（本地磁盘）

直接将文件读写到真实磁盘，通过 `root_dir` 限制访问范围。

> ⚠️ 安全提醒：建议始终使用 `virtual_mode=True` 启用路径沙箱。启用后阻止 `..`、`~` 和 `root_dir` 外的绝对路径。

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend

# 访问本地项目目录
agent = create_deep_agent(
    model="deepseek:deepseek-v4-flash",
    backend=FilesystemBackend(
        root_dir="/mnt/g/ai_code/langchain_study",
        virtual_mode=True,  # 启用路径沙箱
    ),
)

print("Agent 可读写 /mnt/g/ai_code/langchain_study 目录下的文件。")

Agent 可读写 /mnt/g/ai_code/langchain_study 目录下的文件。


### ⚠️ 安全风险（FilesystemBackend）

| 风险 | 说明 |
|------|------|
| 读取敏感文件 | Agent 可读取 API Key、凭据、`.env` 等 |
| SSRF 泄露 | 结合网络工具可能泄露机密 |
| 文件修改不可逆 | 写操作永久生效 |

**建议防护：**
- 启用 HITL（Human-in-the-Loop）中间件审查敏感操作
- 排除机密文件路径
- 生产环境使用 Sandbox
- 始终使用 `virtual_mode=True`

## 5. StoreBackend（LangGraph Store）

基于 LangGraph `BaseStore` 的跨线程持久化存储，适合长期记忆。

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import StoreBackend
from langgraph.store.memory import InMemoryStore

# 本地开发使用 InMemoryStore；LangSmith Deployment 自动注入 store
agent = create_deep_agent(
    model="deepseek:deepseek-v4-flash",
    backend=StoreBackend(
        namespace=lambda rt: (rt.server_info.user.identity,),
    ),
    store=InMemoryStore(),
)

print("StoreBackend：跨线程持久化，适合长期记忆。")

StoreBackend：跨线程持久化，适合长期记忆。


### Namespace 工厂模式

通过 `namespace` 参数控制数据隔离范围：

```python
from deepagents.backends import StoreBackend

# 按用户隔离
backend = StoreBackend(
    namespace=lambda rt: (rt.server_info.user.identity,),
)

# 按 Assistant 共享
backend = StoreBackend(
    namespace=lambda rt: (rt.server_info.assistant_id,),
)

# 按线程隔离
backend = StoreBackend(
    namespace=lambda rt: (rt.execution_info.thread_id,),
)
```

`Runtime` 提供：
- `rt.context` — 用户上下文
- `rt.server_info` — 服务器元数据（assistant ID、graph ID、认证用户）
- `rt.execution_info` — 执行标识（thread ID、run ID、checkpoint ID）

## 6. ContextHubBackend（LangSmith Hub）

将文件持久化存储在 LangSmith Hub repo 中，无需单独配置 LangGraph Store。

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import ContextHubBackend

# 需设置 LANGSMITH_API_KEY 环境变量
# agent = create_deep_agent(
#     model="deepseek:deepseek-v4-flash",
#     backend=ContextHubBackend("my-agent"),
# )

print("ContextHubBackend：在 LangSmith Hub 中持久化文件。")
print("工作原理：首次使用懒加载 Hub repo 树到内存缓存，")
print("写入和编辑操作作为 Hub commits 提交并更新缓存。")

ContextHubBackend：在 LangSmith Hub 中持久化文件。
工作原理：首次使用懒加载 Hub repo 树到内存缓存，
写入和编辑操作作为 Hub commits 提交并更新缓存。


## 7. CompositeBackend（路由组合）

将不同路径路由到不同后端，灵活性最高。默认路径使用 `StateBackend`，特定路径可指向其他后端。

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import CompositeBackend, StateBackend, FilesystemBackend

# 默认路由到 StateBackend，/memories/ 路由到本地磁盘
agent = create_deep_agent(
    model="deepseek:deepseek-v4-flash",
    backend=CompositeBackend(
        default=StateBackend(),
        routes={
            "/memories/": FilesystemBackend(
                root_dir="/mnt/g/ai_code/langchain_study/memories",
                virtual_mode=True,
            ),
        },
    ),
)

print("CompositeBackend 配置完成。")
print("- /memories/* → 本地磁盘 (FilesystemBackend)")
print("- 其他路径 → 线程内存储 (StateBackend)")

CompositeBackend 配置完成。
- /memories/* → 本地磁盘 (FilesystemBackend)
- 其他路径 → 线程内存储 (StateBackend)


### 7.1 结合 FilesystemBackend 的最佳实践

Deep Agents 会自动写入内部数据（`/large_tool_results/`、`/conversation_history/`）。如果单独使用 `FilesystemBackend`，这些内部文件会混入项目目录。推荐使用 `CompositeBackend` 分离：

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import CompositeBackend, StateBackend, FilesystemBackend

agent = create_deep_agent(
    model="deepseek:deepseek-v4-flash",
    backend=CompositeBackend(
        default=StateBackend(),  # 内部数据保留在线程存储中
        routes={
            "/workspace/": FilesystemBackend(
                root_dir="/mnt/g/ai_code/langchain_study",
                virtual_mode=True,
            ),
        },
    ),
)

print("最佳实践：项目文件→本地磁盘，内部数据→StateBackend。")

最佳实践：项目文件→本地磁盘，内部数据→StateBackend。


## 8. LocalShellBackend（本地 Shell）

⚠️ **极度危险：** 赋予 Agent 任意 Shell 执行权限。仅在受控开发环境中使用。

扩展了 `FilesystemBackend`，额外提供 `execute` 工具执行 shell 命令。

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import LocalShellBackend

# 仅在可信的开发环境中使用
agent = create_deep_agent(
    model="deepseek:deepseek-v4-flash",
    backend=LocalShellBackend(
        root_dir=".",
        virtual_mode=True,
        env={"PATH": "/usr/bin:/bin"},
    ),
)

print("LocalShellBackend：文件系统 + Shell 执行。")
print("强烈建议启用 HITL 中间件审查所有操作。")

LocalShellBackend：文件系统 + Shell 执行。
强烈建议启用 HITL 中间件审查所有操作。


### 安全风险（LocalShellBackend）

| 风险 | 说明 |
|------|------|
| 任意命令执行 | 以用户权限运行任意 shell 命令 |
| 机密泄露 | 可读取 API Key、凭据等 |
| 资源无限制 | CPU、内存、磁盘无限制使用 |
| 操作不可逆 | 文件修改和命令执行不可回滚 |

| 适用场景 | 不适用场景 |
|----------|----------|
| 本地开发 CLI 工具 | 生产环境 / Web 服务器 |
| 受信的个人开发环境 | 多租户系统 |
| CI/CD 管道（妥善管理机密） | 处理未受信用户输入 |

## 9. Sandbox（沙箱后端）

在隔离环境中执行代码，提供文件系统工具 + `execute` 工具。支持多种沙箱提供商。

In [ ]:
# Sandbox 后端示例
# from deepagents.backends import sandbox

# agent = create_deep_agent(
#     model="deepseek:deepseek-v4-flash",
#     backend=sandbox,  # 自动选择配置好的沙箱
# )

print("Sandbox 后端：隔离的代码执行环境。")
print("支持的沙箱类型：Modal / Daytona / Deno / 本地 VFS")

Sandbox 后端：隔离的代码执行环境。
支持的沙箱类型：Modal / Daytona / Deno / 本地 VFS


## 10. 权限控制（Permissions）

使用 `FilesystemPermission` 声明式控制 Agent 对文件系统的读写权限。权限在后端调用前评估。

In [ ]:
from deepagents import create_deep_agent, FilesystemPermission
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend

agent = create_deep_agent(
    model="deepseek:deepseek-v4-flash",
    backend=CompositeBackend(
        default=StateBackend(),
        routes={
            "/memories/": StoreBackend(
                namespace=lambda rt: (rt.server_info.user.identity,),
            ),
            "/policies/": StoreBackend(
                namespace=lambda rt: (rt.server_info.assistant_id, "policies"),
            ),
        },
    ),
    permissions=[
        FilesystemPermission("/memories/", "write"),
        FilesystemPermission("/policies/", "read"),
    ],
    store=InMemoryStore(),
)

print("权限控制已配置：")

权限控制已配置：


## 11. 自定义虚拟文件系统

实现 `BackendProtocol` 接口，可将远程存储（S3、Postgres 等）投影为文件系统。

In [ ]:
from deepagents.backends.protocol import (
    BackendProtocol, WriteResult, EditResult,
    LsResult, ReadResult, GrepResult, GlobResult,
)

class S3Backend(BackendProtocol):
    """"将 S3 bucket 映射为文件系统的自定义后端"""
    def __init__(self, bucket: str, prefix: str = ""):
        self.bucket = bucket
        self.prefix = prefix.rstrip("/")

    def _key(self, path: str) -> str:
        return f"{self.prefix}{path}"

    def ls(self, path: str) -> LsResult:
        # 列出 _key(path) 下的对象；构建 FileInfo 条目
        ...

    def read(self, file_path: str, offset: int = 0, limit: int = 2000) -> ReadResult:
        # 获取对象内容
        ...

    def grep(self, pattern: str, path: str | None = None, glob: str | None = None) -> GrepResult:
        # 可选服务端过滤；否则列出后扫描内容
        ...

    def glob(self, pattern: str, path: str = "/") -> GlobResult:
        # 在 keys 上应用 glob 匹配
        ...

    def write(self, file_path: str, content: str) -> WriteResult:
        # 仅创建语义；返回 WriteResult(path=file_path, files_update=None)
        ...

    def edit(self, file_path: str, old_string: str, new_string: str, replace_all: bool = False) -> EditResult:
        # 读取 → 替换 → 写入 → 返回发生次数
        ...

print("S3Backend 骨架定义完成（需实现具体方法）。")

S3Backend 骨架定义完成（需实现具体方法）。


### 设计指南（自定义后端）

- 路径格式：使用绝对路径 `/x/y.txt`
- 高效实现 `ls` 和 `glob`（优先服务端过滤）
- 外部持久化后端返回 `files_update=None`（仅内存状态后端需要返回 files update）
- 使用 `ls` 和 `glob` 作为方法名
- 返回结构化结果类型，错误字段用于缺失文件或无效模式（不要 raise）

---

**参考链接**
- [Deep Agents Backends 文档](https://docs.langchain.com/oss/python/deepagents/backends)
- [LangSmith 可观测性快速开始](https://docs.langchain.com/oss/python/observability)
- [LangSmith Engine](https://docs.langchain.com/oss/python/observability/engine)
- [LangSmith Deployment 生产部署](https://docs.langchain.com/oss/python/deployment)